# Классификация графов с использованием графовых нейронных сетей

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Макрушин С.В. Машинное обучение на графах", Лекции 4-5 "Графовые нейронные сети"
* Документация:
    * https://pytorch-geometric.readthedocs.io/en/latest/modules/utils.html#torch_geometric.utils.from_networkx
    * https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html
    * https://pytorch-geometric.readthedocs.io/en/latest/modules/utils.html?highlight=scatter#torch_geometric.utils.scatter

## Вопросы для совместного обсуждения

In [17]:
import os
import glob
import networkx as nx
import pandas as pd
import torch
import torch.nn.functional as F
import pytorch_lightning as L
import torchmetrics
from IPython.display import display, Markdown
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool, global_add_pool
from torch_geometric.utils import from_networkx
from pytorch_lightning.callbacks import ModelCheckpoint
from typing import Callable
from torch_geometric.utils import scatter
from torch.utils.data import random_split

1\. Обсудите задачу предсказания характеристик графа, подходы к получению векторного представления графа и способы объединения графов в пакеты.

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Реализуйте все описанные методы класса `GraphsDataset`. Используя данный класс, создайте датасет на основе файлов архива `graphs.zip` (можно разархивировать вручную или программно). Выведите на экран количество объектов в датасете. Выведите на экран значения признаков узлов для графа с индексом 0. 

- [ ] Проверено на семинаре

In [18]:
class GraphsDataset(Dataset):
    def __init__(self, root_dir: str):
        super().__init__(root_dir, transform=None, pre_transform=None)
        self.root_dir = root_dir
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        self.file_paths = []
        self.labels =[]
        for cls_name in self.classes:
            cls_dir = os.path.join(root_dir, cls_name)
            files = glob.glob(os.path.join(cls_dir, "*.gml"))
            self.file_paths.extend(files)
            self.labels.extend([self.class_to_idx[cls_name]] * len(files))

    def len(self) -> int:
        return len(self.file_paths)

    def get(self, idx: int) -> Data:
        file_path = self.file_paths[idx]
        label = self.labels[idx]
        G = nx.read_gml(file_path)
        deg_cent = nx.degree_centrality(G)
        clust_coef = nx.clustering(G)
        for node in G.nodes():
            G.nodes[node]['degree_centrality'] = deg_cent[node]
            G.nodes[node]['clustering_coefficient'] = clust_coef[node]
        raw_data = from_networkx(G)
        clean_data = Data(
            edge_index=raw_data.edge_index,
            degree_centrality=raw_data.degree_centrality,
            clustering_coefficient=raw_data.clustering_coefficient,
            num_nodes=raw_data.num_nodes
        )
        clean_data.y = torch.tensor([label], dtype=torch.long)
        return clean_data



In [19]:
dataset = GraphsDataset(root_dir="data/graphs")
print(f"Количество графов в датасете: {len(dataset)}")
graph_0 = dataset[0]
print(f"\nГраф 0: Класс {graph_0.y.item()}")
print("Свойства графа 0:", graph_0)

Количество графов в датасете: 600

Граф 0: Класс 0
Свойства графа 0: Data(edge_index=[2, 72], degree_centrality=[13], clustering_coefficient=[13], num_nodes=13, y=[1])


<p class="task" id="2"></p>

2\. Используя датасет из предыдущего задания, создайте объект `torch_geometric.loader.DataLodaer`. Получите один батч размера 128 при помощи этого объекта и выведите на экран (используйте соответствующие атрибуты и методы):

* количество узлов в графе-батче;
* количество связей в графе-батче;
* количество графов в батче;
* количество узлов в каждом графе батча;

Используя локальную области видимости, создайте в полученном батче атрибут `feat` путем объединения атрибутов `degree_centrality` и `clustering_coefficient`. Выполните readout для графа на основе атрибута `feat`. Выведите размерность полученного тензора на экран.

- [ ] Проверено на семинаре

In [20]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

batch = next(iter(train_loader))

print(f"Количество узлов в графе-батче: {batch.num_nodes}")
print(f"Количество связей в графе-батче: {batch.num_edges}")
print(f"Количество графов в батче: {batch.num_graphs}")

nodes_per_graph = torch.bincount(batch.batch)
print(f"Количество узлов в первых 5 графах батча: {nodes_per_graph[:5].tolist()}")

batch.feat = torch.stack([batch.degree_centrality, batch.clustering_coefficient], dim=-1).float()
readout_tensor = global_mean_pool(batch.feat, batch.batch)

print(f"\nРазмерность тензора признаков узлов (feat): {batch.feat.shape}")
print(f"Размерность тензора после Readout (усреднения): {readout_tensor.shape}")

Количество узлов в графе-батче: 2904
Количество связей в графе-батче: 12444
Количество графов в батче: 128
Количество узлов в первых 5 графах батча: [19, 60, 32, 18, 16]

Размерность тензора признаков узлов (feat): torch.Size([2904, 2])
Размерность тензора после Readout (усреднения): torch.Size([128, 2])


<p class="task" id="3"></p>

3\. Решите задачу классификации графа, используя слои `SAGEConv` и операцию усреднения для процедуры readout. Для обучения используйте стохастический градиентный спуск с размером батча 128. Во время обучения выводите значение функции потерь по эпохам (используйте `torchmetrics`). Вычислите матрицу несоответствий прогнозов и точность обученной модели (используйте `torchmetrics`).  

- [ ] Проверено на семинаре

In [21]:
class GraphClassifierSAGE(L.LightningModule):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int, readout_fn: Callable):
        super().__init__()
        self.save_hyperparameters(ignore=['readout_fn'])
        self.readout_fn = readout_fn
        
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, out_channels)
        
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=out_channels)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=out_channels)
        self.train_confmat = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=out_channels)
        
        self.train_loss_metric = torchmetrics.MeanMetric()
        self.val_loss_metric = torchmetrics.MeanMetric()

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, batch: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = self.readout_fn(x, batch)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin(x)
        return x

    def training_step(self, batch_data, batch_idx):
        x = torch.stack([batch_data.degree_centrality, batch_data.clustering_coefficient], dim=-1).float()
        logits = self(x, batch_data.edge_index, batch_data.batch)
        loss = F.cross_entropy(logits, batch_data.y)
        preds = torch.argmax(logits, dim=1)
        self.train_acc(preds, batch_data.y)
        self.train_confmat(preds, batch_data.y)
        self.train_loss_metric(loss)
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch_data, batch_idx):
        x = torch.stack([batch_data.degree_centrality, batch_data.clustering_coefficient], dim=-1).float()
        logits = self(x, batch_data.edge_index, batch_data.batch)
        loss = F.cross_entropy(logits, batch_data.y)
        preds = torch.argmax(logits, dim=1)
        self.val_acc(preds, batch_data.y)
        self.val_loss_metric(loss)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        epoch_loss = self.train_loss_metric.compute()
        epoch_acc = self.train_acc.compute()
        print(f"Train Epoch {self.current_epoch}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.4f}")
        self.train_loss_metric.reset()
        self.train_acc.reset()

    def on_validation_epoch_end(self):
        epoch_loss = self.val_loss_metric.compute()
        epoch_acc = self.val_acc.compute()
        print(f"Val Epoch {self.current_epoch}: Loss = {epoch_loss:.4f}, Accuracy = {epoch_acc:.4f}")
        self.val_loss_metric.reset()
        self.val_acc.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=0.01, momentum=0.9)
        return optimizer

In [26]:
num_classes = len(dataset.classes)

model_mean = GraphClassifierSAGE(
    in_channels=2,
    hidden_channels=64,
    out_channels=num_classes,
    readout_fn=global_mean_pool
)

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    dirpath='checkpoints',
    filename='task3-best'
)

trainer = L.Trainer(max_epochs=40, enable_progress_bar=False, callbacks=[checkpoint_callback], logger=False)

trainer.fit(model_mean, train_loader, val_loader)

best_model_path = checkpoint_callback.best_model_path
best_model = GraphClassifierSAGE.load_from_checkpoint(
    best_model_path,
    in_channels=2,
    hidden_channels=64,
    out_channels=num_classes,
    readout_fn=global_mean_pool
)

best_model.eval()

with torch.no_grad():
    for batch_data in val_loader:
        batch_data = batch_data.to(best_model.device)        
        x = torch.stack([batch_data.degree_centrality, batch_data.clustering_coefficient], dim=-1).float()
        logits = best_model(x, batch_data.edge_index, batch_data.batch)
        preds = torch.argmax(logits, dim=1)
        best_model.train_confmat(preds, batch_data.y)

conf_matrix = best_model.train_confmat.compute()
print("\nФинальная Матрица Несоответствий на валидационном наборе (Confusion Matrix):")
print(conf_matrix.cpu().numpy())

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv1             │ SAGEConv                  │    320 │ train │     0 │
│ 1 │ conv2             │ SAGEConv                  │  8.3 K │ train │     0 │
│ 2 │ lin               │ Linear                    │    195 │ train │     0 │
│ 3 │ train_acc         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 4 │ val_acc           │ MulticlassAccuracy        │      0 │ train │     0 │
│ 5 │ train_confmat     │ MulticlassConfusionMatrix │      0 │ train │     0 │
│ 6 │ train_loss_metric │ MeanMetric                │      0 │ train │     0 │
│ 7 │ val_loss_metric   │ MeanMetric                │      0 │ train │     0 │
└───┴───────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 8.8 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.8 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Val Epoch 0: Loss = 1.1014, Accuracy = 0.5000
Val Epoch 0: Loss = 1.0979, Accuracy = 0.3750
Train Epoch 0: Loss = 1.0933, Accuracy = 0.3771
Val Epoch 1: Loss = 1.0928, Accuracy = 0.3000
Train Epoch 1: Loss = 1.0744, Accuracy = 0.4104
Val Epoch 2: Loss = 1.0863, Accuracy = 0.2750
Train Epoch 2: Loss = 1.0750, Accuracy = 0.4042
Val Epoch 3: Loss = 1.0761, Accuracy = 0.2750
Train Epoch 3: Loss = 1.0576, Accuracy = 0.4042
Val Epoch 4: Loss = 1.0609, Accuracy = 0.3667
Train Epoch 4: Loss = 1.0530, Accuracy = 0.4187
Val Epoch 5: Loss = 1.0443, Accuracy = 0.3917
Train Epoch 5: Loss = 1.0340, Accuracy = 0.4479
Val Epoch 6: Loss = 1.0301, Accuracy = 0.4167
Train Epoch 6: Loss = 1.0168, Accuracy = 0.5292
Val Epoch 7: Loss = 1.0148, Accuracy = 0.4333
Train Epoch 7: Loss = 0.9935, Accuracy = 0.5562
Val Epoch 8: Loss = 0.9971, Accuracy = 0.5000
Train Epoch 8: Loss = 0.9760, Accuracy = 0.5938
Val Epoch 9: Loss = 0.9825, Accuracy = 0.5333
Train Epoch 9: Loss = 0.9722, Accuracy = 0.5479
Val Epoch 10: 

`Trainer.fit` stopped: `max_epochs=40` reached.


Val Epoch 39: Loss = 0.6909, Accuracy = 0.6917
Train Epoch 39: Loss = 0.6874, Accuracy = 0.7042

Финальная Матрица Несоответствий на валидационном наборе (Confusion Matrix):
[[26  0  7]
 [ 1 34 12]
 [ 5 12 23]]


<p class="task" id="4"></p>

4\. Повторите решение задачи 3, сравнив разные функции агрегации для проведения операции readout. Выведите результаты в виде таблицы:

Выведите результат в виде таблицы:

| Readout op    | Loss | Acc |
|-----------|------------------|-----------------|
| sum |                  |                 | 
| mean  |                  |                 |
| max   |                  |                 |
| min   |                  |                 |

- [ ] Проверено на семинаре

In [28]:
def custom_min_pool(x, batch):
    return scatter(x, batch, dim=0, reduce='min')

readout_options = {
    'sum': global_add_pool,
    'mean': global_mean_pool,
    'max': global_max_pool, 
    'min': custom_min_pool
}

results =[]

for name, r_fn in readout_options.items():
    print(f"\nОбучение с Readout: {name.upper()}")
    
    model = GraphClassifierSAGE(
        in_channels=2, 
        hidden_channels=64, 
        out_channels=num_classes, 
        readout_fn=r_fn
    )
    
    checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        mode='min',
        save_top_k=1,
        dirpath='checkpoints',
        filename=f'{name}-best'
    )
    
    trainer = L.Trainer(
        max_epochs=40, 
        enable_progress_bar=False, 
        callbacks=[checkpoint_callback],
        logger=False 
    )
    
    trainer.fit(model, train_loader, val_loader)

    best_model_path = checkpoint_callback.best_model_path
    
    best_model = GraphClassifierSAGE.load_from_checkpoint(
        best_model_path,
        in_channels=2,
        hidden_channels=64,
        out_channels=num_classes,
        readout_fn=r_fn
    )
    
    best_model.eval()
    best_model.val_acc.reset()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in val_loader:
            batch_data = batch_data.to(best_model.device)
            x = torch.stack([batch_data.degree_centrality, batch_data.clustering_coefficient], dim=-1).float().to(best_model.device)
            logits = best_model(x, batch_data.edge_index, batch_data.batch)
            loss = F.cross_entropy(logits, batch_data.y)
            preds = torch.argmax(logits, dim=1)
            best_model.val_acc(preds, batch_data.y)
            total_loss += loss.item() * batch_data.num_graphs

    best_loss = total_loss / len(val_loader.dataset)
    best_acc = best_model.val_acc.compute().item()
    
    results.append({
        "Readout op": name,
        "Loss": round(best_loss, 4),
        "Acc": round(best_acc, 4)
    })

results_df = pd.DataFrame(results)
display(Markdown(results_df.to_markdown(index=False)))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



Обучение с Readout: SUM


┏━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name              ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ conv1             │ SAGEConv                  │    320 │ train │     0 │
│ 1 │ conv2             │ SAGEConv                  │  8.3 K │ train │     0 │
│ 2 │ lin               │ Linear                    │    195 │ train │     0 │
│ 3 │ train_acc         │ MulticlassAccuracy        │      0 │ train │     0 │
│ 4 │ val_acc           │ MulticlassAccuracy        │      0 │ train │     0 │
│ 5 │ train_confmat     │ MulticlassConfusionMatrix │      0 │ train │     0 │
│ 6 │ train_loss_metric │ MeanMetric                │      0 │ train │     0 │
│ 7 │ val_loss_metric   │ MeanMetric                │      0 │ train │     0 │
└───┴───────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 8.8 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.8 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Val Epoch 0: Loss = 1.6860, Accuracy = 0.3333
Val Epoch 0: Loss = 1.0255, Accuracy = 0.4000
Train Epoch 0: Loss = 2.0454, Accuracy = 0.3729
Val Epoch 1: Loss = 0.8362, Accuracy = 0.5667
Train Epoch 1: Loss = 0.9825, Accuracy = 0.4833
Val Epoch 2: Loss = 0.6893, Accuracy = 0.6000
Train Epoch 2: Loss = 0.8550, Accuracy = 0.5292
Val Epoch 3: Loss = 0.7298, Accuracy = 0.5917
Train Epoch 3: Loss = 0.8165, Accuracy = 0.5500
Val Epoch 4: Loss = 0.6728, Accuracy = 0.6000
Train Epoch 4: Loss = 0.7416, Accuracy = 0.5938
Val Epoch 5: Loss = 0.6474, Accuracy = 0.6333
Train Epoch 5: Loss = 0.7027, Accuracy = 0.6062
Val Epoch 6: Loss = 0.6039, Accuracy = 0.6167
Train Epoch 6: Loss = 0.6667, Accuracy = 0.6167
Val Epoch 7: Loss = 0.5955, Accuracy = 0.6417
Train Epoch 7: Loss = 0.6643, Accuracy = 0.6438
Val Epoch 8: Loss = 0.5751, Accuracy = 0.6833
Train Epoch 8: Loss = 0.6764, Accuracy = 0.6479
Val Epoch 9: Loss = 0.5621, Accuracy = 0.6750
Train Epoch 9: Loss = 0.6422, Accuracy = 0.6896
Val Epoch 10: 

...

Trainable params: 8.8 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 8.8 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 14                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Val Epoch 0: Loss = 1.0873, Accuracy = 0.5667
Val Epoch 0: Loss = 1.0823, Accuracy = 0.4250
Train Epoch 0: Loss = 1.0901, Accuracy = 0.3833
Val Epoch 1: Loss = 1.0759, Accuracy = 0.3333
Train Epoch 1: Loss = 1.0825, Accuracy = 0.3812
Val Epoch 2: Loss = 1.0704, Accuracy = 0.2833
Train Epoch 2: Loss = 1.0655, Accuracy = 0.4021
Val Epoch 3: Loss = 1.0640, Accuracy = 0.2833
Train Epoch 3: Loss = 1.0322, Accuracy = 0.4083
Val Epoch 4: Loss = 1.0536, Accuracy = 0.3333
Train Epoch 4: Loss = 1.0203, Accuracy = 0.4750
Val Epoch 5: Loss = 1.0381, Accuracy = 0.5083
Train Epoch 5: Loss = 1.0058, Accuracy = 0.4854
Val Epoch 6: Loss = 1.0178, Accuracy = 0.5333
Train Epoch 6: Loss = 0.9941, Accuracy = 0.5583
Val Epoch 7: Loss = 0.9974, Accuracy = 0.6000
Train Epoch 7: Loss = 0.9758, Accuracy = 0.5792
Val Epoch 8: Loss = 0.9774, Accuracy = 0.6083
Train Epoch 8: Loss = 0.9482, Accuracy = 0.6167
Val Epoch 9: Loss = 0.9581, Accuracy = 0.6250
Train Epoch 9: Loss = 0.9378, Accuracy = 0.6229
Val Epoch 10: 

`Trainer.fit` stopped: `max_epochs=40` reached.


Val Epoch 39: Loss = 0.6350, Accuracy = 0.7500
Train Epoch 39: Loss = 0.6264, Accuracy = 0.7542


| Readout op   |   Loss |    Acc |
|:-------------|-------:|-------:|
| sum          | 0.3539 | 0.8833 |
| mean         | 0.6877 | 0.7    |
| max          | 0.5653 | 0.8    |
| min          | 0.635  | 0.75   |